# MCMC mesuré : Metropolis–Hastings et Gibbs en C#

**Objectifs** : construire deux échantillonneurs, comparer des approximations et diagnostiquer les chaînes. **Prérequis** : probabilités conditionnelles, variance et C#. **Durée** : 90 minutes. [Version Python](Probas-MCMC-Echantillonnage-Metropolis-Python.ipynb) · [Index Probas](../README.md).

MathNet.Numerics fournit les lois normales et statistiques ; MH, Gibbs et les diagnostics sont construits ici. Les générateurs C# et NumPy diffèrent : on compare aux vérités analytiques, pas les trajectoires bit à bit.


In [1]:
#r "nuget: MathNet.Numerics, 5.0.0"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text;
using System.Globalization;
using MathNet.Numerics.Distributions;
using MathNet.Numerics.Statistics;
using Microsoft.DotNet.Interactive.Formatting;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages MathNet.Numerics, 5.0.0

## Cibles et approximations

T1 est une gaussienne de moyenne (1, −1), variances 1 et corrélation 0,8. T2 est un mélange équiprobable de normales centrées en ±2,5 d'écart-type 0,5 : sa moyenne vaut 0 et sa variance 6,5. Le produit des marges de T1 perd sa corrélation ; une normale au seul mode positif de T2 omet l'autre mode.


In [2]:
double LogT1(double[] x) { double a=x[0]-1, b=x[1]+1; return -(a*a-1.6*a*b+b*b)/(2*(1-0.8*0.8)); }
double LogT2(double[] x) { double a=-0.5*Math.Pow((x[0]-2.5)/0.5,2), b=-0.5*Math.Pow((x[0]+2.5)/0.5,2); double m=Math.Max(a,b); return m+Math.Log(Math.Exp(a-m)+Math.Exp(b-m)); }
const int N=10000, Burn=1000;
int[] seeds={0,1,7,42};
Console.WriteLine("T1 : moyenne (1, -1), covariance [[1, .8], [.8, 1]] ; T2 : moyenne 0, variance 6.5");


T1 : moyenne (1, -1), covariance [[1, .8], [.8, 1]] ; T2 : moyenne 0, variance 6.5


## Moteurs

La proposition gaussienne est symétrique : le test `log(u) ≤ log p(proposition) − log p(état)` évite de calculer un rapport exponentiel. Un rejet conserve l'état ; la chauffe est écartée. Gibbs tire les conditionnelles exactes de variance 1−ρ² et utilise immédiatement la nouvelle première coordonnée.


In [3]:
double[][] MH(Func<double[],double> logp, double[] start, int n, int burn, double step, Random rng) {
    var x=(double[])start.Clone(); double lp=logp(x); var outp=new double[n][];
    for(int t=0;t<n+burn;t++) {
        var proposal=x.Select(v => v+Normal.Sample(rng,0,step)).ToArray();
        double next=logp(proposal);
        if(Math.Log(rng.NextDouble()) <= next-lp) { x=proposal; lp=next; }
        if(t>=burn) outp[t-burn]=(double[])x.Clone();
    }
    return outp;
}
double[][] Gibbs(int n,int burn,Random rng) {
    double x=1+Normal.Sample(rng,0,1), y=-1+Normal.Sample(rng,0,1); var output=new double[n][];
    for(int t=0;t<n+burn;t++) {
        x=Normal.Sample(rng,1+0.8*(y+1),0.6);
        y=Normal.Sample(rng,-1+0.8*(x-1),0.6);
        if(t>=burn) output[t-burn]=new[]{x,y};
    }
    return output;
}
Console.WriteLine("MH et Gibbs définis.");


MH et Gibbs définis.


## Diagnostics classiques

Pour m chaînes de longueur n, `W` est la moyenne des variances intra-chaînes et `B = n × variance des moyennes`. `R-hat = sqrt(((n−1)W/n + B/n)/W)`. C'est le diagnostic classique **non scindé**, distinct des variantes modernes scindées et normalisées par rangs. Pour W nul, on renvoie l'infini si les moyennes diffèrent, sinon NaN. L'ESS ci-dessous est une heuristique tronquée au premier coefficient d'autocorrélation négatif, sans garantie de borne.


In [4]:
double Rhat(double[][] chains) {
    int m=chains.Length; if(m<2) return double.NaN; int n=chains[0].Length;
    if(n<2 || chains.Any(c => c.Length!=n || c.Any(v => !double.IsFinite(v)))) return double.NaN;
    double w=chains.Average(c => c.Variance());
    double b=n*chains.Select(c=>c.Average()).Variance();
    if(w==0) return b>0 ? double.PositiveInfinity : double.NaN;
    return Math.Sqrt(((n-1.0)/n*w+b/n)/w);
}
double Ess(double[] x,int maxLag=4000) {
    double mean=x.Average(); var d=x.Select(v=>v-mean).ToArray(); double denom=d.Sum(v=>v*v);
    if(denom<=0) return double.NaN; double tau=1;
    for(int k=1;k<Math.Min(maxLag,x.Length-1);k++) {
        double sum=0; for(int i=k;i<x.Length;i++) sum+=d[i]*d[i-k];
        double r=sum/denom; if(r<0) break; tau+=2*r;
    }
    return x.Length/Math.Max(1,tau);
}
var fixture=new[]{new[]{-3.0,-1,-3,-1},new[]{1.0,3,1,3}};
if(Math.Abs(Rhat(fixture)-Math.Sqrt(27.0/4))>1e-12 || !double.IsPositiveInfinity(Rhat(new[]{new[]{0.0,0},new[]{1.0,1}})) || !double.IsNaN(Rhat(new[]{new[]{0.0,0},new[]{0.0,0}}))) throw new Exception("Diagnostic incorrect");
var same=new[]{new[]{-1.0,1,-1,1},new[]{-1.0,1,-1,1}};
if(Math.Abs(Rhat(same)-Math.Sqrt(0.75))>1e-12) throw new Exception("Cas chaînes identiques incorrect");
Console.WriteLine($"Oracle analytique : R-hat={Rhat(fixture):F6} (attendu √6.75), cas W=0 et chaînes identiques vérifiés.");


Oracle analytique : R-hat=2.598076 (attendu √6.75), cas W=0 et chaînes identiques vérifiés.


## Expérience T1 fixe : ρ = 0,8

Quatre chaînes utilisent les graines 0, 1, 7, 42, 10 000 échantillons et 1 000 pas de chauffe. Les points de départ sont tirés près de la moyenne ; le pas MH vaut 1. Les résultats aléatoires sont comparés à la covariance analytique et au produit indépendant.


In [5]:
var mh=seeds.Select(s=>{var r=new Random(s); return MH(LogT1,new[]{1+Normal.Sample(r,0,1),-1+Normal.Sample(r,0,1)},N,Burn,1,r);}).ToArray();
var gib=seeds.Select(s=>Gibbs(N,Burn,new Random(s))).ToArray();
var rngMF=new Random(0); var mf=Enumerable.Range(0,N).Select(i=>new[]{Normal.Sample(rngMF,1,1),Normal.Sample(rngMF,-1,1)}).ToArray();
double Corr(double[][] a)=>Correlation.Pearson(a.Select(v=>v[0]),a.Select(v=>v[1]));
double[] X(double[][] a)=>a.Select(v=>v[0]).ToArray();
double[] Y(double[][] a)=>a.Select(v=>v[1]).ToArray();
var pooled=mh.SelectMany(c=>c).ToArray();
Console.WriteLine($"MH : moyennes ({X(pooled).Average():F3}, {Y(pooled).Average():F3}), variances ({X(pooled).Variance():F3}, {Y(pooled).Variance():F3}), corrélation {Corr(pooled):F3}");
Console.WriteLine($"Gibbs : corrélation {Corr(gib.SelectMany(c=>c).ToArray()):F3} ; mean-field : {Corr(mf):F3} ; vérité : 0.8");
Console.WriteLine($"MH ESS moyen/chaîne {mh.Average(c=>Ess(X(c))):F0}, R-hat coordonnée 1 {Rhat(mh.Select(X).ToArray()):F4}");
if(Math.Abs(Corr(pooled)-0.8)>0.08 || Math.Abs(Corr(mf))>0.08 || Math.Abs(Corr(gib.SelectMany(c=>c).ToArray())-0.8)>0.08) throw new Exception("T1 hors tolérance 0.08");


MH : moyennes (1.010, -0.987), variances (1.045, 1.026), corrélation 0.805


Gibbs : corrélation 0.799 ; mean-field : -0.001 ; vérité : 0.8


MH ESS moyen/chaîne 404, R-hat coordonnée 1 1.0013


### Interprétation

La tolérance 0,08 est large devant les fluctuations attendues sur quatre chaînes mais détecte une perte majeure de corrélation. Un R-hat proche de 1 est un indice, non une preuve de convergence.

## Expérience T2 fixe : modes ±2,5

Les quatre chaînes partent du mode positif, avec pas 0,8. L'occupation et les moyennes individuelles sont essentielles : une variance groupée proche de 6,5 peut masquer des chaînes bloquées dans des modes distincts.


In [6]:
var t2=seeds.Select(s=>MH(LogT2,new[]{2.5},N,Burn,0.8,new Random(s)).Select(v=>v[0]).ToArray()).ToArray();
var all=t2.SelectMany(c=>c).ToArray(); var rngNaive=new Random(0); var naive=Enumerable.Range(0,N).Select(i=>Normal.Sample(rngNaive,2.5,0.5)).ToArray();
Console.WriteLine($"T2 groupé : moyenne {all.Average():F3}, variance {all.Variance():F3}, fraction positive {all.Count(v=>v>0)/(double)all.Length:F3}, R-hat {Rhat(t2):F3}");
for(int i=0;i<seeds.Length;i++) Console.WriteLine($"graine {seeds[i],2} : moyenne {t2[i].Average():F3}, fraction positive {t2[i].Count(v=>v>0)/(double)N:F3}, ESS {Ess(t2[i]):F0}");
Console.WriteLine($"Mode unique : moyenne {naive.Average():F3}, variance {naive.Variance():F3} ; vérité : moyenne 0, variance 6.5, fraction positive 0.5");
Console.WriteLine(Rhat(t2)>1.01 ? "R-hat signale une non-convergence." : "R-hat ne démontre pas le mélange : inspecter les traces.");


T2 groupé : moyenne 1.254, variance 4.937, fraction positive 0.750, R-hat 5.122


graine  0 : moyenne -2.496, fraction positive 0.000, ESS 2040


graine  1 : moyenne 2.505, fraction positive 1.000, ESS 1645


graine  7 : moyenne 2.492, fraction positive 1.000, ESS 1943


graine 42 : moyenne 2.518, fraction positive 1.000, ESS 2171


Mode unique : moyenne 2.505, variance 0.253 ; vérité : moyenne 0, variance 6.5, fraction positive 0.5


R-hat signale une non-convergence.


## Figures des données observées

Les histogrammes utilisent les mêmes bornes et sont normalisés en densité. Les traces permettent de distinguer une exploration de chaque chaîne d'une simple agrégation de chaînes bloquées.


In [7]:
public record PlotSvg(string Content);
Formatter.Register<PlotSvg>((v,w)=>w.Write(v.Content), "text/html");
string F(double x)=>x.ToString("F2",CultureInfo.InvariantCulture);
string Head(string title)=>$"<svg xmlns='http://www.w3.org/2000/svg' width='800' height='300' viewBox='0 0 800 300'><rect width='800' height='300' fill='white'/><text x='15' y='22' font-size='15'>{title}</text>";
string Scatter(double[][] a,double[][] b) { var s=new StringBuilder(Head("T1 : MH bleu, mean-field rouge")); foreach(var pair in new[]{(a,"#2369b0"),(b,"#c54436")}) foreach(var v in pair.Item1.Where((v,i)=>i%20==0)) { double x=400+(v[0]-1)*60,y=160-(v[1]+1)*55; if(x>5&&x<795&&y>30&&y<290) s.Append($"<circle cx='{F(x)}' cy='{F(y)}' r='1.7' fill='{pair.Item2}' fill-opacity='.35'/>"); } return s.Append("</svg>").ToString(); }
string Hist(double[] a,double[] b) { var s=new StringBuilder(Head("T2 : densités normalisées, MH bleu et mode unique rouge")); int bins=80; double lo=-5, width=10.0/bins; foreach(var pair in new[]{(a,"#2369b0"),(b,"#c54436")}) { var counts=new int[bins]; foreach(double x in pair.Item1) { int k=(int)Math.Floor((x-lo)/width); if(k>=0&&k<bins) counts[k]++; } for(int k=0;k<bins;k++) { double density=counts[k]/(pair.Item1.Length*width), height=density*220; s.Append($"<rect x='{F(40+k*9)}' y='{F(270-height)}' width='9' height='{F(height)}' fill='{pair.Item2}' fill-opacity='.4'/>"); }} return s.Append("</svg>").ToString(); }
string Trace(double[][] a) { var s=new StringBuilder(Head("T2 : traces par chaîne, graines 0 / 1 / 7 / 42")); string[] colors={"#2369b0","#c54436","#16814c","#8a48aa"}; for(int j=0;j<a.Length;j++) { s.Append($"<polyline fill='none' stroke='{colors[j]}' stroke-width='.7' points='"); for(int i=0;i<a[j].Length;i+=20) s.Append($"{F(40+i*0.072)},{F(155-a[j][i]*20)} "); s.Append("'/>"); } return s.Append("</svg>").ToString(); }
display(new PlotSvg(Scatter(mh[0],mf)));
display(new PlotSvg(Hist(all,naive)));
display(new PlotSvg(Trace(t2)));


T1 : MH bleu, mean-field rouge

T2 : densités normalisées, MH bleu et mode unique rouge

T2 : traces par chaîne, graines 0 / 1 / 7 / 42 <polyline fill='none' stroke='#2369b0' stroke-width='.7' points='40.00,206.05 41.44,190.12 42.88,178.58 44.32,213.28 45.76,199.33 47.20,206.74 48.64,225.66 50.08,223.48 51.52,203.48 52.96,187.28 54.40,215.11 55.84,210.02 57.28,198.91 58.72,209.21 60.16,197.54 61.60,210.78 63.04,214.46 64.48,202.93 65.92,212.67 67.36,206.64 68.80,217.28 70.24,209.22 71.68,208.04 73.12,208.30 74.56,201.29 76.00,202.34 77.44,207.96 78.88,195.66 80.32,207.31 81.76,209.46 83.20,199.75 84.64,208.97 86.08,205.82 87.52,210.14 88.96,194.91 90.40,194.17 91.84,221.87 93.28,171.48 94.72,198.64 96.16,188.72 97.60,200.64 99.04,204.64 100.48,206.24 101.92,201.55 103.36,201.24 104.80,213.99 106.24,207.77 107.68,206.04 109.12,224.31 110.56,175.81 112.00,207.20 113.44,210.39 114.88,195.90 116.32,189.64 117.76,206.84 119.20,210.73 120.64,200.04 122.08,218.41 123.52,204.19 124.96,208.28 126.40,209.88 127.84,211.03 129.28,206.93 130.72,189.36 132.16,203.82 133.60,218.76 135.04,220.46 136.48,222.99 137.92,204.77 139.36,187.92 140.80,199.59 142.24,207.80 143.68,221.01 145.12,230.44 146.56,191.42 148.00,208.61 149.44,198.45 150.88,210.01 152.32,203.49 153.76,201.06 155.20,198.03 156.64,208.20 158.08,206.69 159.52,228.35 160.96,200.22 162.40,193.78 163.84,199.38 165.28,211.50 166.72,199.84 168.16,202.01 169.60,219.18 171.04,205.15 172.48,204.68 173.92,209.48 175.36,196.23 176.80,210.08 178.24,202.00 179.68,189.91 181.12,230.20 182.56,204.41 184.00,193.91 185.44,209.01 186.88,209.17 188.32,205.28 189.76,213.68 191.20,209.75 192.64,192.31 194.08,207.43 195.52,213.80 196.96,201.01 198.40,199.95 199.84,200.78 201.28,211.94 202.72,188.24 204.16,214.86 205.60,200.60 207.04,205.24 208.48,201.40 209.92,207.26 211.36,200.36 212.80,205.77 214.24,195.79 215.68,210.56 217.12,195.74 218.56,206.34 220.00,211.90 221.44,203.10 222.88,202.70 224.32,214.87 225.76,213.44 227.20,202.56 228.64,212.77 230.08,202.22 231.52,211.38 232.96,203.42 234.40,197.09 235.84,207.81 237.28,196.74 238.72,207.52 240.16,187.33 241.60,182.93 243.04,201.50 244.48,195.86 245.92,212.98 247.36,202.62 248.80,217.13 250.24,194.28 251.68,208.13 253.12,210.54 254.56,207.70 256.00,209.45 257.44,217.38 258.88,202.11 260.32,194.10 261.76,206.77 263.20,187.61 264.64,212.76 266.08,208.11 267.52,204.15 268.96,208.55 270.40,201.34 271.84,201.84 273.28,214.45 274.72,203.66 276.16,205.51 277.60,185.53 279.04,197.62 280.48,199.01 281.92,212.18 283.36,203.87 284.80,200.43 286.24,198.59 287.68,207.67 289.12,203.77 290.56,196.41 292.00,213.71 293.44,183.36 294.88,185.73 296.32,198.60 297.76,193.56 299.20,208.44 300.64,204.27 302.08,200.92 303.52,206.34 304.96,222.96 306.40,218.29 307.84,209.16 309.28,194.52 310.72,197.98 312.16,196.28 313.60,203.60 315.04,205.33 316.48,217.34 317.92,209.50 319.36,207.24 320.80,190.29 322.24,203.28 323.68,213.68 325.12,216.63 326.56,211.92 328.00,201.74 329.44,209.56 330.88,202.48 332.32,196.63 333.76,201.27 335.20,215.01 336.64,201.42 338.08,210.11 339.52,198.34 340.96,200.67 342.40,218.41 343.84,200.20 345.28,214.97 346.72,204.06 348.16,214.25 349.60,193.66 351.04,190.66 352.48,202.92 353.92,209.73 355.36,217.91 356.80,205.45 358.24,198.49 359.68,188.96 361.12,200.05 362.56,204.00 364.00,205.29 365.44,194.71 366.88,202.87 368.32,210.36 369.76,207.51 371.20,192.42 372.64,227.81 374.08,203.33 375.52,195.35 376.96,199.42 378.40,208.36 379.84,205.30 381.28,210.19 382.72,190.47 384.16,201.29 385.60,212.01 387.04,199.80 388.48,206.11 389.92,195.08 391.36,202.45 392.80,216.47 394.24,201.10 395.68,195.39 397.12,219.46 398.56,210.00 400.00,175.33 401.44,205.65 402.88,191.61 404.32,197.46 405.76,211.45 407.20,197.17 408.64,212.59 410.08,182.95 411.52,192.74 412.96,207.41 414.40,197.64 415.84,220.18 417.28,212.85 418.72,207.47 420.16,204.40 421.60,209.99 423.04,204.08 424.48,218.26 425.92,226.47 427.36,207.94 428.80,205.36 430.24,212.10 431.68,190.78 433.12,194.36 434.56,204.31 436.00,190.34 437.44,194.10 438.88,204.33 440.32,217.83 441.76,202.01 443.20,204

### Lecture

Les nuages distinguent dépendance et indépendance. Les histogrammes comparent les masses sur une échelle commune ; les traces et les résultats par chaîne ci-dessus éclairent la métastabilité. Une valeur agrégée seule ne prouve pas le mélange.

## Exercice 1 — Probabilité d'acceptation

Pour une proposition symétrique, retourner `min(1, exp(logpNouveau − logpCourant))`. Éviter un dépassement lorsque la proposition est plus probable.


In [8]:
double? ProbAccept(double oldLog,double newLog) { return null; /* TODO */ }
var p1=ProbAccept(0,1); var p2=ProbAccept(0,Math.Log(0.5));
if(p1.HasValue && Math.Abs(p1.Value-1)>1e-12 || p2.HasValue && Math.Abs(p2.Value-0.5)>1e-12) throw new Exception("Exercice 1");
Console.WriteLine("Exercice 1 à compléter.");


Exercice 1 à compléter.


## Exercice 2 — ESS depuis une ACF

Pour le tableau donné, tronquer au premier coefficient négatif et calculer `N/(1+2 somme rho[k])`.


In [9]:
double? EssFromAcf(double[] rho,int n) { return null; /* TODO */ }
var e=EssFromAcf(new[]{1.0,0.5,0.25,-0.1},5);
if(e.HasValue && Math.Abs(e.Value-2)>1e-12) throw new Exception("Exercice 2");
Console.WriteLine("Exercice 2 à compléter.");


Exercice 2 à compléter.


## Exercice 3 — Taux d'acceptation

Mesurer la fraction de propositions acceptées. Le pas nul repropose toujours le point courant ; avec un pas très grand, les propositions de cette graine sont rejetées.


In [10]:
double? AcceptanceRate(double step,int n,int seed) { return null; /* TODO : cible T1, départ (1,-1) */ }
var rate=AcceptanceRate(0,500,0);
if(rate.HasValue && Math.Abs(rate.Value-1)>1e-12) throw new Exception("Exercice 3");
Console.WriteLine("Exercice 3 à compléter.");


Exercice 3 à compléter.


## À retenir

- MH utilise les log-densités ; Gibbs tire ici des conditionnelles exactes.
- La factorisation perd la dépendance ; une approximation à un seul mode omet l'autre.
- Une variance groupée plausible ne suffit pas à conclure au mélange. R-hat classique et ESS heuristique ont des limites.

## Vérifiez votre compréhension

Pourquoi un R-hat proche de 1 peut-il masquer des chaînes bloquées dans le même mode ? Pourquoi une variance groupée peut-elle être proche de 6,5 sans mélange individuel ?

## Pour aller plus loin

Metropolis et al. (1953), Hastings (1970), Gelman et al., *Bayesian Data Analysis*, 3e éd., §11 ; [carnets PyMC](../PyMC/).
